# DeBERTa Fine-Tuning for Team Morale Prediction

This notebook fine-tunes `microsoft/deberta-v3-base` for **regression** on a custom football morale dataset.  
The model predicts a **morale score (0–10)** based on a natural language description of a team's recent form, results, and context.

> **Note:** Designed to run on **Google Colab** (GPU required). The trained model is exported as a `.zip` and stored locally under `models/deberta-morale-final/`.

## 1. Installation

Install required libraries: `transformers`, `datasets`, `torch`, and `scikit-learn`.

In [4]:
!pip install transformers==4.44.0
!pip install datasets torch scikit-learn

## 2. Imports

In [5]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, Value
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from huggingface_hub import login

## 3. Hugging Face Authentication

Log in to Hugging Face Hub to access the DeBERTa model weights.

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
login(token=os.environ.get("HF_TOKEN"))

## 4. Data Loading

Mount Google Drive and load train/validation splits.  
Each dataset contains two columns: `text` (natural language morale description) and `label` (morale score 1–10).

In [7]:
from google.colab import drive
drive.mount('/content/drive')

train_df = pd.read_csv('/content/drive/MyDrive/sports_prediction/train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/sports_prediction/val.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
print(f"Train NaN labels: {train_df['label'].isna().sum()} / {len(train_df)}")
print(f"Val NaN labels:   {val_df['label'].isna().sum()} / {len(val_df)}")

Train NaN labels: 0 / 800
Val NaN labels:   0 / 200


### Data Cleaning

Drop any rows with missing `text` or `label` values to avoid training errors.

In [9]:
train_df = train_df.dropna(subset=['label', 'text'])
val_df = val_df.dropna(subset=['label', 'text'])

### Label Normalization

Scale labels from `[1, 10]` to `[0.1, 1.0]` by dividing by `LABEL_MAX = 10.0`.  
Keeping values in a small range improves regression stability during training.

In [10]:
LABEL_MAX = 10.0

train_df['label'] = train_df['label'].astype(np.float32)/LABEL_MAX
val_df['label'] = val_df['label'].astype(np.float32)/LABEL_MAX

In [11]:
print(f"\nPo czyszczeniu — Train: {len(train_df)}, Val: {len(val_df)}")
print(f"Label range train: {train_df['label'].min():.2f} – {train_df['label'].max():.2f}")
print(f"Label range val:   {val_df['label'].min():.2f} – {val_df['label'].max():.2f}")


Po czyszczeniu — Train: 800, Val: 200
Label range train: 0.10 – 1.00
Label range val:   0.10 – 1.00


## 5. Tokenizer & Tokenization

Load the DeBERTa tokenizer and define a tokenization function.  
All sequences are truncated or padded to a fixed length of **256 tokens**.

In [12]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch['text'], truncation = True, padding = "max_length", max_length = 256)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


### Dataset Preparation

Convert DataFrames to Hugging Face `Dataset` objects, apply tokenization in batches, rename the label column, and cast labels to `float32`.

In [13]:
train_dataset = Dataset.from_pandas(train_df).map(tokenize, batched = True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize, batched = True)

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [14]:
train_dataset = train_dataset.cast_column("labels", Value("float32"))
val_dataset = val_dataset.cast_column("labels", Value("float32"))

Casting the dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

In [15]:
train_dataset.set_format(type = "torch", columns = ["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type = "torch", columns = ["input_ids", "attention_mask", "labels"])

### Data Validation

Sanity check — confirm there are no `NaN` or `Inf` values in the label tensors before training starts.

In [16]:
train_labels = torch.tensor(train_dataset['labels'])
val_labels = torch.tensor(val_dataset['labels'])
print(f"\nTensor NaN check — train: {torch.isnan(train_labels).sum()}, val: {torch.isnan(val_labels).sum()}")
print(f"Tensor Inf check — train: {torch.isinf(train_labels).sum()}, val: {torch.isinf(val_labels).sum()}")


Tensor NaN check — train: 0, val: 0
Tensor Inf check — train: 0, val: 0


In [17]:
print(f"Label dtype train: {train_dataset[0]['labels'].dtype}")
print(f"Label dtype val: {val_dataset[0]['labels'].dtype}")

Label dtype train: torch.float32
Label dtype val: torch.float32


## 6. Model

Load `deberta-v3-base` as a sequence classifier with `num_labels=1` (single continuous output for regression).  
The classification head weights are randomly initialized and will be trained from scratch.

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels = 1)
print(f"\nGPU is available: {torch.cuda.is_available()}")

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



GPU is available: True


In [19]:
from transformers import TrainingArguments, Trainer

## 7. Evaluation Metrics

Define **MSE** and **MAE** computed on the original (denormalized) scale.  
The `Trainer` will use `mae` to select the best checkpoint at the end of training.

In [20]:
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = predictions.squeeze()

  pred_orig = predictions * LABEL_MAX
  labels_orig = labels * LABEL_MAX
  mse = np.mean((pred_orig - labels_orig) ** 2)
  mae = np.mean(np.abs(pred_orig - labels_orig))
  return {"mse": float(mse), "mae": float(mae)}

## 8. Training Configuration

Set up `TrainingArguments` and initialize the `Trainer`.  
Key hyperparameters: **5 epochs**, batch size **16**, learning rate **2e-5**, warmup ratio **0.1**, best model loaded at end.

In [21]:
args = TrainingArguments(
    output_dir = "./deberta-morale",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    metric_for_best_model = "mae",
    logging_steps=10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 5,
    warmup_ratio=0.1,
    weight_decay = 0.01,
    learning_rate = 2e-5,
    max_grad_norm = 1.0,
    load_best_model_at_end = True,
    fp16=False,
    bf16=False,
    greater_is_better= False,
)
trainer = Trainer(
    model = model,
    args = args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    compute_metrics = compute_metrics,
)

## 9. Training

Start fine-tuning. Evaluation runs after each epoch; the best checkpoint (lowest validation MAE) is automatically restored at the end.

In [22]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Mse,Mae
1,0.016300,0.007315,0.731495,0.682083
2,0.007200,0.002794,0.279394,0.423703
3,0.006500,0.002762,0.276204,0.430660
4,0.003600,0.002444,0.244375,0.400391
5,0.004000,0.001667,0.166706,0.320853


TrainOutput(global_step=250, training_loss=0.03687903933227062, metrics={'train_runtime': 484.5034, 'train_samples_per_second': 8.256, 'train_steps_per_second': 0.516, 'total_flos': 526226823168000.0, 'train_loss': 0.03687903933227062, 'epoch': 5.0})

## 10. Save & Export

Save the fine-tuned model and tokenizer to `./deberta-morale-final`, then zip and download the folder for local use.

In [24]:
trainer.save_model("./deberta-morale-final")
tokenizer.save_pretrained('./deberta-morale-final')

import shutil
from google.colab import files

shutil.make_archive("deberta-morale-final", 'zip', "./deberta-morale-final")
files.download("deberta-morale-final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>